In [1]:
import os
import json
import time
from typing import Optional, Dict, Any
import supervision as sv
import numpy as np
from loguru import logger
from ultralytics import YOLO

from video_processor import VideoProcessor

In [ ]:
MODEL_PATH = 'runs/detect/train1/weights/best.pt'
device = 'cuda'

In [4]:
model = YOLO(MODEL_PATH).to(device=device)

In [5]:
video_path = 'videos_35min/part_01.mp4'

video_processor = VideoProcessor(
    device=device,
    cuda_timeout=10800.0,
    mps_timeout=10800.0,
    cpu_timeout=10800.0
)

if not await video_processor.ensure_video_readable(video_path):
    raise HTTPException(
        status_code=400,
        detail="Video file is not readable or corrupted"
    )

2025-05-05 18:01:34.597 | INFO     | video_processor:__init__:29 - Video processor initialized with cuda device, timeout: 10800.0s


In [6]:
start_time = time.time()

tracker = sv.ByteTrack(track_activation_threshold=0.05, minimum_matching_threshold=0.7, lost_track_buffer=600, frame_rate=30)
tracking_data = {"frames": []}

async for frame_number, frame in video_processor.stream_frames(video_path):
    player_result = model(frame, imgsz=736, conf=0.05, verbose=False)[0]
            
    detections = sv.Detections.from_ultralytics(player_result)
    detections = detections.with_nms(threshold=0.35)
    detections = tracker.update_with_detections(detections)

    frame_data = {
        "frame_number": int(frame_number),  
        "objects": [
            {
                "id": int(tracker_id),
                "bbox": [float(x) for x in bbox],
                "class_id": int(class_id)
            }
            for tracker_id, bbox, class_id in zip(
                detections.tracker_id,
                detections.xyxy,
                detections.class_id
            )
        ] if detections and detections.tracker_id is not None else []
    }
    tracking_data["frames"].append(frame_data)

    if frame_number % 100 == 0:
        elapsed = time.time() - start_time
        fps = frame_number / elapsed if elapsed > 0 else 0
        logger.info(f"Processed {frame_number} frames in {elapsed:.1f}s ({fps:.2f} fps)")

processing_time = time.time() - start_time
total_frames = len(tracking_data["frames"])
fps = total_frames / processing_time if processing_time > 0 else 0

logger.info(
    f"Completed processing {total_frames} frames in {processing_time:.1f}s "
    f"({fps:.2f} fps) on {device} device"
)


2025-05-05 18:01:37.668 | INFO     | __main__:<module>:33 - Processed 0 frames in 1.7s (0.00 fps)
2025-05-05 18:01:39.596 | INFO     | __main__:<module>:33 - Processed 100 frames in 3.7s (27.28 fps)
2025-05-05 18:01:39.874 | INFO     | video_processor:read_frames:67 - Completed processing 2560 frames in 3.9s on cuda device
2025-05-05 18:01:41.244 | INFO     | __main__:<module>:33 - Processed 200 frames in 5.3s (37.64 fps)
2025-05-05 18:01:42.895 | INFO     | __main__:<module>:33 - Processed 300 frames in 7.0s (43.07 fps)
2025-05-05 18:01:44.566 | INFO     | __main__:<module>:33 - Processed 400 frames in 8.6s (46.32 fps)
2025-05-05 18:01:46.280 | INFO     | __main__:<module>:33 - Processed 500 frames in 10.4s (48.31 fps)
2025-05-05 18:01:48.097 | INFO     | __main__:<module>:33 - Processed 600 frames in 12.2s (49.32 fps)
2025-05-05 18:01:49.921 | INFO     | __main__:<module>:33 - Processed 700 frames in 14.0s (50.03 fps)
2025-05-05 18:01:51.781 | INFO     | __main__:<module>:33 - Proces

In [7]:
import nest_asyncio
import asyncio
import cv2
import json
import time
from pathlib import Path

output_video_path = "test_part1.mp4"

nest_asyncio.apply()

frames_data = tracking_data["frames"]

# Open the video file
cap = cv2.VideoCapture(str(video_path))

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)

# Initialize VideoWriter to save the video with bounding boxes
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video_path, fourcc, fps, frame_size)

# Define colors for different classes
COLORS = {
    0: (0, 255, 0),   # Green - Regular Players
}

def plot_bounding_boxes(frame, frame_data):
    for obj in frame_data.get("objects", []):
        bbox = obj["bbox"]
        class_id = obj["class_id"]
        tracker_id = obj.get("id", None)  # Get tracker ID if present

        x_min, y_min, x_max, y_max = map(int, bbox)
        color = COLORS.get(class_id, (255, 255, 255))  # Default to white

        # Draw the bounding box
        frame = cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 2)

        # Draw the tracker ID if it exists
        if tracker_id is not None:
            label = f"ID: {tracker_id}"
            frame = cv2.putText(frame, label, (x_min, y_min - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return frame

# Process and save frames with bounding boxes
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print(f"End of video or error reading frame at {frame_count}.")
        break

    frame_data = frames_data[frame_count]
    if frame_data:
        frame = plot_bounding_boxes(frame, frame_data)

    # Write the frame with bounding boxes to the output video
    out.write(frame)
    frame_count += 1

cap.release()
out.release()
print(f"Video with bounding boxes saved to: {output_video_path}")

End of video or error reading frame at 2560.
Video with bounding boxes saved to: test_part1.mp4
